# Mid-fusion Multimodal Notebook (Image + Text)
This notebook demonstrates a mid-fusion model that encodes image volumes and report text separately, then combines their representations at the fusion layer.

## Architecture Overview
- **Image Encoder**: 3D CNN to encode MRI volume patches
- **Text Encoder**: Transformer-based encoder for clinical report text
- **Fusion Layer**: Concatenate or learned fusion of embeddings
- **Classification Head**: Predict abnormality labels

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
import numpy as np
import pandas as pd
from pathlib import Path

# Set random seeds for reproducibility
pl.seed_everything(42)
print(f'PyTorch version: {torch.__version__}')
print(f'PyTorch Lightning version: {pl.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

## Define Mid-Fusion Multimodal Model

In [ ]:
class ImageEncoder(nn.Module):
    """3D CNN encoder for MRI volumes."""
    def __init__(self, in_channels=1, out_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1)
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


class TextEncoder(nn.Module):
    """Simple text encoder using embedding + pooling."""
    def __init__(self, vocab_size=1000, embed_dim=128, out_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, 256, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(512, out_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        x = torch.cat([h_n[0], h_n[1]], dim=1)
        x = self.fc(x)
        return x


class MidFusionMultimodal(pl.LightningModule):
    """Mid-fusion multimodal model combining image and text."""
    def __init__(self, num_classes=3, image_dim=256, text_dim=256, lr=1e-3):
        super().__init__()
        self.image_encoder = ImageEncoder(out_dim=image_dim)
        self.text_encoder = TextEncoder(out_dim=text_dim)
        self.fusion_fc = nn.Sequential(
            nn.Linear(image_dim + text_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
        self.criterion = nn.CrossEntropyLoss()
        self.lr = lr

    def forward(self, images, texts):
        image_emb = self.image_encoder(images)
        text_emb = self.text_encoder(texts)
        fused = torch.cat([image_emb, text_emb], dim=1)
        logits = self.fusion_fc(fused)
        return logits

    def training_step(self, batch, batch_idx):
        images, texts, labels = batch
        logits = self(images, texts)
        loss = self.criterion(logits, labels)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        images, texts, labels = batch
        logits = self(images, texts)
        loss = self.criterion(logits, labels)
        self.log('val_loss', loss)

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.lr)
        return opt

## Training example
Wire up dataset, dataloader and Lightning Trainer with TensorBoard and EarlyStopping similar to the baseline.

**Note:** Adapt the dataset loading and batch format to match your actual data structure.

In [ ]:
# Example training setup (requires actual data)
# from torch.utils.data import DataLoader, TensorDataset
#
# # Dummy data for demonstration
# batch_size = 16
# image_data = torch.randn(100, 1, 32, 64, 64)  # (N, C, D, H, W)
# text_data = torch.randint(0, 1000, (100, 50))  # (N, seq_len)
# labels = torch.randint(0, 3, (100,))
#
# dataset = TensorDataset(image_data, text_data, labels)
# train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
# val_loader = DataLoader(dataset, batch_size=batch_size)
#
# model = MidFusionMultimodal(num_classes=3, lr=1e-3)
# logger = TensorBoardLogger('tb_logs', name='mid_fusion')
# early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min')
#
# trainer = pl.Trainer(
#     max_epochs=20,
#     logger=logger,
#     callbacks=[early_stop],
#     accelerator='gpu' if torch.cuda.is_available() else 'cpu',
#     devices=1
# )
#
# trainer.fit(model, train_loader, val_loader)